PySpark Feature Engineering

Replicates the PostgreSQL feature engineering layer (Phase 2) using PySpark
to demonstrate the same logic at distributed scale.

**Why this matters:** PostgreSQL runs on a single machine. At production scale
(millions of customers), the same CTEs and window functions would run across
a distributed Spark cluster instead. This notebook proves the logic ports over
cleanly from SQL to PySpark.

**Dataset:** IBM Telco Customer Churn (7,043 rows, 21 columns)

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChurnFeatureEngineering") \
    .getOrCreate()

print(spark.version)

4.0.4


In [2]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (2).csv


## 1. Load Data into a Spark DataFrame

Unlike pandas, Spark DataFrames are lazy — operations aren't executed immediately.
Spark builds a plan and only computes when you call an action like `.show()` or `.count()`.

In [3]:
df = spark.read.csv('WA_Fn-UseC_-Telco-Customer-Churn.csv', header=True, inferSchema=True)

print("Row count:", df.count())
df.printSchema()

Row count: 7043
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



## 2. Clean TotalCharges and Convert Churn to Binary

Same fix as SQL: `TotalCharges` has blank strings for 11 customers with zero tenure.
We cast it to a numeric type, which turns blanks into nulls automatically.

In [4]:
from pyspark.sql.functions import col, when, trim, expr

df = df.withColumn("TotalCharges", expr("try_cast(trim(TotalCharges) AS DOUBLE)"))
df = df.filter(col("TotalCharges").isNotNull())
df = df.withColumn("churn_value", when(col("Churn") == "Yes", 1).otherwise(0))

print("Row count after cleaning:", df.count())

Row count after cleaning: 7032


## 3. Feature Engineering — Tenure Bucket and Spend Tier

Same logic as the SQL `CASE WHEN` statements from Phase 2, written using PySpark's `when()`.

In [5]:
df = df.withColumn("tenure_bucket",
    when(col("tenure") <= 12, "0-12m")
    .when(col("tenure") <= 24, "12-24m")
    .otherwise("24m+")
)

df = df.withColumn("spend_tier",
    when(col("MonthlyCharges") < 35, "Low")
    .when(col("MonthlyCharges") < 65, "Medium")
    .otherwise("High")
)

df.select("tenure", "tenure_bucket", "MonthlyCharges", "spend_tier").show(10)

+------+-------------+--------------+----------+
|tenure|tenure_bucket|MonthlyCharges|spend_tier|
+------+-------------+--------------+----------+
|     1|        0-12m|         29.85|       Low|
|    34|         24m+|         56.95|    Medium|
|     2|        0-12m|         53.85|    Medium|
|    45|         24m+|          42.3|    Medium|
|     2|        0-12m|          70.7|      High|
|     8|        0-12m|         99.65|      High|
|    22|       12-24m|          89.1|      High|
|    10|        0-12m|         29.75|       Low|
|    28|         24m+|         104.8|      High|
|    62|         24m+|         56.15|    Medium|
+------+-------------+--------------+----------+
only showing top 10 rows


## 4. Window Functions — Replicating SQL's Rolling Metrics

This is the layer that proves Spark can do everything SQL's CTEs and window
functions could do, but at distributed scale. We calculate average monthly
charges per contract type using `Window` partitioning — the PySpark
equivalent of `PARTITION BY` in SQL.

In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, round as spark_round

# Define a window partitioned by Contract type
contract_window = Window.partitionBy("Contract")

# Calculate average monthly charges within each contract type
df = df.withColumn(
    "avg_charges_by_contract",
    spark_round(avg("MonthlyCharges").over(contract_window), 2)
)

df.select("customerID", "Contract", "MonthlyCharges", "avg_charges_by_contract").show(10)

+----------+--------------+--------------+-----------------------+
|customerID|      Contract|MonthlyCharges|avg_charges_by_contract|
+----------+--------------+--------------+-----------------------+
|7590-VHVEG|Month-to-month|         29.85|                   66.4|
|3668-QPYBK|Month-to-month|         53.85|                   66.4|
|9237-HQITU|Month-to-month|          70.7|                   66.4|
|9305-CDSKC|Month-to-month|         99.65|                   66.4|
|1452-KIOVK|Month-to-month|          89.1|                   66.4|
|6713-OKOMC|Month-to-month|         29.75|                   66.4|
|7892-POOKP|Month-to-month|         104.8|                   66.4|
|9763-GRSKD|Month-to-month|         49.95|                   66.4|
|0280-XJGEX|Month-to-month|         103.7|                   66.4|
|5129-JLPIS|Month-to-month|         105.5|                   66.4|
+----------+--------------+--------------+-----------------------+
only showing top 10 rows


## 5. Save Final Feature Table

Export the engineered features as Parquet — the standard format for
distributed data at scale (columnar storage, compressed, much faster
than CSV for large datasets). This mirrors what the PostgreSQL `features`
table represents, but in a format built for big data pipelines.

In [8]:
final_df = df.select(
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
    "tenure", "tenure_bucket", "Contract", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "spend_tier",
    "avg_charges_by_contract", "churn_value"
)

final_df.write.mode("overwrite").parquet("telco_features_pyspark.parquet")

final_df.toPandas().to_csv("telco_features_pyspark.csv", index=False)

print("Saved successfully")
final_df.show(5)

Saved successfully
+----------+------+-------------+-------+----------+------+-------------+--------------+--------------------+--------------+------------+----------+-----------------------+-----------+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|tenure_bucket|      Contract|       PaymentMethod|MonthlyCharges|TotalCharges|spend_tier|avg_charges_by_contract|churn_value|
+----------+------+-------------+-------+----------+------+-------------+--------------+--------------------+--------------+------------+----------+-----------------------+-----------+
|7590-VHVEG|Female|            0|    Yes|        No|     1|        0-12m|Month-to-month|    Electronic check|         29.85|       29.85|       Low|                   66.4|          0|
|3668-QPYBK|  Male|            0|     No|        No|     2|        0-12m|Month-to-month|        Mailed check|         53.85|      108.15|    Medium|                   66.4|          1|
|9237-HQITU|Female|            0|     No|        No|    

In [9]:
from google.colab import files
files.download('telco_features_pyspark.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

This notebook replicated the Phase 2 PostgreSQL feature engineering pipeline
using PySpark DataFrames and window functions. The same tenure buckets, spend
tiers, and partition-based aggregations were reproduced with identical results
(7,032 rows after cleaning), demonstrating that this pipeline is portable to
a distributed Spark environment for production-scale data.

**Key equivalences demonstrated:**
| SQL | PySpark |
|---|---|
| `CASE WHEN` | `when().otherwise()` |
| `WHERE` | `.filter()` |
| `PARTITION BY ... OVER` | `Window.partitionBy().over()` |
| `SELECT ... AS` | `.withColumn()` |